# Alzheimer's MRI Stage Classifier: training notebook

Fine-tunes a pre-trained ResNet50 on a four-class Alzheimer's MRI dataset (NonDemented, VeryMildDemented, MildDemented, ModerateDemented).

**How to run (Google Colab):**
1. Runtime > Change runtime type > **T4 GPU**.
2. Upload `archive.zip` (the Kaggle dataset) to the Colab Files panel and wait for the upload to finish.
3. Run the cells from top to bottom.

The trained weights are saved as `alz_resnet50.pth`, which the Streamlit app (`app.py`) loads.

## 1. Unzip the dataset

In [ ]:
!mkdir -p /content/data
!unzip -q /content/archive.zip -d /content/data
!find /content/data -maxdepth 3 -type d

## 2. Load the data and split it 70% train, 15% validation, 15% test

In [ ]:
import os, numpy as np, torch, torch.nn as nn
from PIL import Image
from torchvision import transforms, models
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print("Device:", device)

DATA_DIR = '/content/data'
norm = lambda s: s.lower().replace('_', '').replace(' ', '')

parents = [dp for dp, dn, fn in os.walk(DATA_DIR) if any(norm(d) == 'nondemented' for d in dn)]
print("Found class folders in:", parents)

class_names = sorted({d for p in parents for d in os.listdir(p) if os.path.isdir(os.path.join(p, d))})
print("Classes:", class_names)

paths, labels = [], []
for p in parents:
    for ci, c in enumerate(class_names):
        folder = os.path.join(p, c)
        if os.path.isdir(folder):
            for f in os.listdir(folder):
                if f.lower().endswith(('.jpg', '.jpeg', '.png')):
                    paths.append(os.path.join(folder, f)); labels.append(ci)
print("Total images:", len(paths))

mean, std = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]
train_tf = transforms.Compose([transforms.Grayscale(3), transforms.Resize((224, 224)),
    transforms.RandomRotation(10), transforms.RandomAffine(0, translate=(0.05, 0.05)),
    transforms.ToTensor(), transforms.Normalize(mean, std)])
eval_tf = transforms.Compose([transforms.Grayscale(3), transforms.Resize((224, 224)),
    transforms.ToTensor(), transforms.Normalize(mean, std)])

class MRIDataset(Dataset):
    def __init__(self, p, l, tf): self.p, self.l, self.tf = p, l, tf
    def __len__(self): return len(self.p)
    def __getitem__(self, i): return self.tf(Image.open(self.p[i]).convert('L')), self.l[i]

p_tr, p_tmp, l_tr, l_tmp = train_test_split(paths, labels, test_size=0.30, stratify=labels, random_state=42)
p_va, p_te, l_va, l_te = train_test_split(p_tmp, l_tmp, test_size=0.50, stratify=l_tmp, random_state=42)

train_dl = DataLoader(MRIDataset(p_tr, l_tr, train_tf), batch_size=32, shuffle=True, num_workers=2)
val_dl   = DataLoader(MRIDataset(p_va, l_va, eval_tf), batch_size=64, num_workers=2)
test_dl  = DataLoader(MRIDataset(p_te, l_te, eval_tf), batch_size=64, num_workers=2)

counts = np.bincount(l_tr)
print("Training images per class:", dict(zip(class_names, counts)))
weights = torch.tensor(counts.sum() / (len(counts) * counts), dtype=torch.float).to(device)

## 3. Build the pre-trained model (transfer learning)

In [ ]:
model = models.resnet50(weights=models.ResNet50_Weights.DEFAULT)
model.fc = nn.Sequential(nn.Dropout(0.3), nn.Linear(model.fc.in_features, len(class_names)))
model = model.to(device)

criterion = nn.CrossEntropyLoss(weight=weights)   # class weights handle the imbalance
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=8)

## 4. Train

In [ ]:
def run_epoch(dl, train=True):
    model.train(train)
    total, correct, loss_sum = 0, 0, 0.0
    with torch.set_grad_enabled(train):
        for x, y in dl:
            x, y = x.to(device), y.to(device)
            out = model(x)
            loss = criterion(out, y)
            if train:
                optimizer.zero_grad(); loss.backward(); optimizer.step()
            loss_sum += loss.item() * len(y)
            correct += (out.argmax(1) == y).sum().item()
            total += len(y)
    return loss_sum / total, correct / total

best_val = 0
for epoch in range(8):
    tl, ta = run_epoch(train_dl, True)
    vl, va = run_epoch(val_dl, False)
    scheduler.step()
    print(f"Epoch {epoch+1}: train acc {ta:.3f} | val acc {va:.3f} | val loss {vl:.3f}")
    if va > best_val:
        best_val = va
        torch.save(model.state_dict(), 'alz_resnet50.pth')
print("Best val acc:", best_val)

## 5. Download the trained weights

In [ ]:
from google.colab import files
files.download('alz_resnet50.pth')

## 6. Evaluate on the test set

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt, seaborn as sns

model.load_state_dict(torch.load('alz_resnet50.pth'))
model.eval()
y_true, y_pred = [], []
with torch.no_grad():
    for x, y in test_dl:
        out = model(x.to(device))
        y_pred += out.argmax(1).cpu().tolist()
        y_true += y.tolist()

print(classification_report(y_true, y_pred, target_names=class_names))
cm = confusion_matrix(y_true, y_pred)
sns.heatmap(cm, annot=True, fmt='d', xticklabels=class_names, yticklabels=class_names, cmap='Blues')
plt.xlabel('Predicted'); plt.ylabel('Actual'); plt.show()